# Expiry-Week Calendar Spread Backtest

**Strategy:** Enter a calendar spread on the Sunday opening of expiration week,
close on Thursday (expiry day) near session close.

- **Sell** the front-month ATM option (expiring this week)
- **Buy** the back-month ATM option (next monthly expiry) at the same strike

**Edge:** The front-month option's theta decay accelerates sharply during the
final week, while the back-month (4+ weeks to expiry) decays slowly.
If the underlying stays near the strike, the spread widens and we profit.

**Universe:** TA-35 monthly options, 2014–2023.

**Pricing:** Best bid/ask midpoint from quote snapshots.

| Parameter | Value |
|---|---|
| Entry window | Sunday of expiration week, timestamp ≤ 09:45 |
| Exit window | Expiry Thursday, timestamp ≥ 17:10 |
| Strike | ATM (closest to median spot at entry) |
| Option type | Calls (configurable) |
| Legs | Short front-month + Long back-month |

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta

DUCKDB_PATH = str(Path.home() / "fg3-data" / "maof.duckdb")
EXPIRATIONS_CSV = Path("..") / "ta35_monthly_expirations_2014_2025.csv"

# --- Configurable parameters ---
OPTION_TYPE    = "C"          # "C" for calls, "P" for puts
ENTRY_TIME_MAX = "09450000"   # Sunday: quotes until 09:45:00.00
EXIT_TIME_MIN  = "17100000"   # Thursday: quotes from 17:10:00.00 onward
DATA_YEARS     = range(2014, 2024)

con = duckdb.connect(DUCKDB_PATH, read_only=True)
print(f"Connected to {DUCKDB_PATH}")
con.execute("SHOW TABLES").fetchdf()

## Build Expiration Pairs

Read the TA-35 monthly expirations calendar, pair each front-month expiry with
the next month's expiry (back-month). Derive the entry Sunday (expiry − 4 days)
for each pair.

In [ ]:
expiries_all = pd.read_csv(EXPIRATIONS_CSV, parse_dates=["expiry_date"])
expiries = (
    expiries_all
    .loc[expiries_all["year"].isin(DATA_YEARS), "expiry_date"]
    .sort_values()
    .reset_index(drop=True)
)

# Back-month for the last front-month may fall outside DATA_YEARS
all_sorted = expiries_all["expiry_date"].sort_values().reset_index(drop=True)

pairs = []
for i in range(len(expiries)):
    front = expiries[i]
    # Find the next expiry in the full calendar (may be outside DATA_YEARS)
    later = all_sorted[all_sorted > front]
    if later.empty:
        continue
    back = later.iloc[0]

    entry_sunday = front - timedelta(days=4)
    if entry_sunday.weekday() != 6:  # pandas: Sunday = 6
        continue

    pairs.append({
        "front_expiry": front,
        "back_expiry": back,
        "entry_sunday": entry_sunday,
        "exit_thursday": front,
    })

pairs_df = pd.DataFrame(pairs)
print(f"Built {len(pairs_df)} expiration-week calendar-spread opportunities")
print(f"First: {pairs_df.iloc[0]['entry_sunday'].date()} -> "
      f"{pairs_df.iloc[0]['exit_thursday'].date()}")
print(f"Last:  {pairs_df.iloc[-1]['entry_sunday'].date()} -> "
      f"{pairs_df.iloc[-1]['exit_thursday'].date()}")
pairs_df.head()

## Load Quote Data

Pull quotes for entry Sundays and exit Thursdays only,
restricted to the front- and back-month expiry dates.
This avoids the full-table scan that timed out in the weekend notebook.

In [ ]:
entry_dates = sorted(pairs_df["entry_sunday"].unique())
exit_dates  = sorted(pairs_df["exit_thursday"].unique())
all_dates   = sorted(set(entry_dates) | set(exit_dates))
all_expiries = sorted(
    set(pairs_df["front_expiry"].tolist() + pairs_df["back_expiry"].tolist())
)

def _sql_date_list(dates):
    return ", ".join(f"'{pd.Timestamp(d).strftime('%Y-%m-%d')}'" for d in dates)

date_sql   = _sql_date_list(all_dates)
expiry_sql = _sql_date_list(all_expiries)

query = f"""
SELECT
    date,
    timestamp,
    strike,
    expiry_date,
    bid_price_1,
    ask_price_1,
    underlying_index,
    (bid_price_1 + ask_price_1) / 2.0 AS midpoint
FROM quotes
WHERE option_type = '{OPTION_TYPE}'
  AND bid_price_1 > 0
  AND ask_price_1 > 0
  AND underlying_index IS NOT NULL
  AND underlying_index > 0
  AND date IN ({date_sql})
  AND expiry_date IN ({expiry_sql})
ORDER BY date, timestamp, strike
"""

df = con.execute(query).fetchdf()
df["date"] = pd.to_datetime(df["date"])
df["expiry_date"] = pd.to_datetime(df["expiry_date"])

print(f"Loaded {len(df):,} {OPTION_TYPE} quotes across {df['date'].nunique()} trading days")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Distinct expiry_dates in data: {df['expiry_date'].nunique()}")
df.head()

## Strategy Engine

For each expiration pair:
1. **Sunday open:** find the median spot, pick ATM strike, record midpoints
   for front-month and back-month at that strike. Entry spread = back − front.
2. **Thursday close (expiry):** find the same two contracts, record midpoints.
   Exit spread = back − front.
3. **P&L** = exit spread − entry spread (spread widening = profit).

If the front-month is illiquid at Thursday close (no two-sided quote in the
late window), fall back to the last two-sided quote on that Thursday, or to
intrinsic value.

In [ ]:
entry_set = set(pd.to_datetime(pairs_df["entry_sunday"]))
exit_set  = set(pd.to_datetime(pairs_df["exit_thursday"]))

sun_early = df[(df["date"].isin(entry_set)) & (df["timestamp"] <= ENTRY_TIME_MAX)]
thu_late  = df[(df["date"].isin(exit_set))  & (df["timestamp"] >= EXIT_TIME_MIN)]
thu_all   = df[df["date"].isin(exit_set)]  # fallback pool

sun_by_date     = dict(list(sun_early.groupby("date")))
thu_by_date     = dict(list(thu_late.groupby("date")))
thu_all_by_date = dict(list(thu_all.groupby("date")))


def _last_mid(pool, strike, expiry):
    """Last midpoint for (strike, expiry) in a quote pool, or None."""
    sub = pool[(pool["strike"] == strike) & (pool["expiry_date"] == expiry)]
    if sub.empty:
        return None
    return sub.sort_values("timestamp").iloc[-1]["midpoint"]


def _first_mid(pool, strike, expiry):
    """First midpoint for (strike, expiry) in a quote pool, or None."""
    sub = pool[(pool["strike"] == strike) & (pool["expiry_date"] == expiry)]
    if sub.empty:
        return None
    return sub.sort_values("timestamp").iloc[0]["midpoint"]


def _intrinsic(spot, strike):
    if OPTION_TYPE == "C":
        return max(0.0, spot - strike)
    return max(0.0, strike - spot)


trades_list = []
skipped = {
    "no_entry_quotes": 0,
    "no_front_entry": 0,
    "no_back_entry": 0,
    "no_exit_quotes": 0,
    "no_front_exit": 0,
    "no_back_exit": 0,
}

for _, pair in pairs_df.iterrows():
    entry_date = pair["entry_sunday"]
    exit_date  = pair["exit_thursday"]
    front_exp  = pair["front_expiry"]
    back_exp   = pair["back_expiry"]

    # --- Entry: Sunday near open ---
    sun_q = sun_by_date.get(entry_date)
    if sun_q is None or sun_q.empty:
        skipped["no_entry_quotes"] += 1
        continue

    spot_entry = sun_q["underlying_index"].median()
    strikes = sun_q["strike"].unique()
    atm_strike = int(strikes[np.argmin(np.abs(strikes - spot_entry))])

    front_entry_mid = _first_mid(sun_q, atm_strike, front_exp)
    if front_entry_mid is None:
        skipped["no_front_entry"] += 1
        continue

    back_entry_mid = _first_mid(sun_q, atm_strike, back_exp)
    if back_entry_mid is None:
        skipped["no_back_entry"] += 1
        continue

    entry_spread = back_entry_mid - front_entry_mid

    # --- Exit: Thursday near close ---
    thu_q = thu_by_date.get(exit_date)

    front_exit_mid = None
    if thu_q is not None and not thu_q.empty:
        front_exit_mid = _last_mid(thu_q, atm_strike, front_exp)
    if front_exit_mid is None:
        pool = thu_all_by_date.get(exit_date)
        if pool is not None:
            front_exit_mid = _last_mid(pool, atm_strike, front_exp)
    if front_exit_mid is None:
        pool = thu_all_by_date.get(exit_date)
        spot_exit_fb = pool["underlying_index"].median() if pool is not None else spot_entry
        front_exit_mid = _intrinsic(spot_exit_fb, atm_strike)

    back_exit_mid = None
    if thu_q is not None and not thu_q.empty:
        back_exit_mid = _last_mid(thu_q, atm_strike, back_exp)
    if back_exit_mid is None:
        pool = thu_all_by_date.get(exit_date)
        if pool is not None:
            back_exit_mid = _last_mid(pool, atm_strike, back_exp)
    if back_exit_mid is None:
        skipped["no_back_exit"] += 1
        continue

    exit_spread = back_exit_mid - front_exit_mid
    pnl = exit_spread - entry_spread

    exit_pool = thu_all_by_date.get(exit_date)
    spot_exit = exit_pool["underlying_index"].median() if exit_pool is not None else spot_entry

    trades_list.append({
        "entry_date": entry_date,
        "exit_date": exit_date,
        "strike": atm_strike,
        "front_expiry": front_exp,
        "back_expiry": back_exp,
        "spot_entry": round(spot_entry, 2),
        "spot_exit": round(spot_exit, 2),
        "front_entry_mid": front_entry_mid,
        "back_entry_mid": back_entry_mid,
        "front_exit_mid": front_exit_mid,
        "back_exit_mid": back_exit_mid,
        "entry_spread": entry_spread,
        "exit_spread": exit_spread,
        "pnl": pnl,
    })

results = pd.DataFrame(trades_list)
if not results.empty:
    results["pnl_pct"] = (
        results["pnl"] / results["entry_spread"].abs().clip(lower=0.1) * 100
    )
print(f"Executed {len(results)} calendar-spread trades")
print(f"Skipped: {skipped}")

## Results

In [ ]:
if not results.empty:
    show = results.copy()
    for col in ["entry_date", "exit_date", "front_expiry", "back_expiry"]:
        show[col] = pd.to_datetime(show[col]).dt.date
    num_cols = [
        "front_entry_mid", "back_entry_mid",
        "front_exit_mid", "back_exit_mid",
        "entry_spread", "exit_spread", "pnl", "pnl_pct",
    ]
    show[num_cols] = show[num_cols].round(1)

    print(f"Total trades: {len(show)}\n")
    print("First 10 trades:")
    display(show.head(10))
    print("\nLast 10 trades:")
    display(show.tail(10))
else:
    print("No trades executed.")

## Summary Statistics

In [ ]:
if not results.empty:
    pnl = results["pnl"]
    cum = pnl.cumsum()
    drawdown = cum - cum.cummax()
    wins = (pnl > 0).sum()

    is_loss = (pnl <= 0).astype(int)
    groups = (is_loss != is_loss.shift()).cumsum()
    streaks = is_loss * (is_loss.groupby(groups).cumcount() + 1)

    sharpe = pnl.mean() / pnl.std() * np.sqrt(12) if pnl.std() > 0 else np.nan

    stats = {
        "Total Trades": len(pnl),
        "Total P&L (ILS)": f"{pnl.sum():,.1f}",
        "Mean P&L / Trade": f"{pnl.mean():.2f}",
        "Median P&L / Trade": f"{pnl.median():.2f}",
        "Std Dev": f"{pnl.std():.2f}",
        "Win Rate": f"{wins}/{len(pnl)} ({100 * wins / len(pnl):.1f}%)",
        "Best Trade": f"{pnl.max():.1f}",
        "Worst Trade": f"{pnl.min():.1f}",
        "Sharpe (annualized)": f"{sharpe:.2f}",
        "Max Drawdown (ILS)": f"{drawdown.min():.1f}",
        "Longest Losing Streak": int(streaks.max()),
    }

    for k, v in stats.items():
        print(f"{k:>25s}:  {v}")

## Charts

In [ ]:
if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    dates = pd.to_datetime(results["entry_date"])
    cum_pnl = results["pnl"].cumsum()

    axes[0].plot(dates, cum_pnl, linewidth=1.2, color="steelblue")
    axes[0].fill_between(dates, cum_pnl, alpha=0.15, color="steelblue")
    axes[0].axhline(0, color="grey", linewidth=0.5, linestyle="--")
    axes[0].set_title("Calendar Spread: Cumulative P&L", fontsize=14)
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Cumulative P&L (ILS / contract)")
    axes[0].grid(True, alpha=0.3)

    pnl = results["pnl"]
    axes[1].hist(pnl, bins=40, edgecolor="white", alpha=0.7, color="steelblue")
    axes[1].axvline(pnl.mean(), color="red", linestyle="--", lw=1.2,
                    label=f"Mean: {pnl.mean():.1f}")
    axes[1].axvline(pnl.median(), color="orange", linestyle="--", lw=1.2,
                    label=f"Median: {pnl.median():.1f}")
    axes[1].axvline(0, color="grey", linewidth=0.8)
    axes[1].set_title("P&L Distribution per Trade", fontsize=14)
    axes[1].set_xlabel("P&L (ILS / contract)")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

In [ ]:
if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    results["year"] = pd.to_datetime(results["entry_date"]).dt.year
    annual = results.groupby("year")["pnl"].agg(["sum", "count", "mean"])
    colors = ["forestgreen" if v >= 0 else "indianred" for v in annual["sum"]]
    axes[0].bar(annual.index, annual["sum"], color=colors, edgecolor="white")
    axes[0].axhline(0, color="grey", linewidth=0.5)
    axes[0].set_title("Annual P&L", fontsize=13)
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Total P&L (ILS)")
    axes[0].grid(True, alpha=0.3, axis="y")

    dates = pd.to_datetime(results["entry_date"])
    cum_pnl = results["pnl"].cumsum()
    dd = cum_pnl - cum_pnl.cummax()
    axes[1].fill_between(dates, dd, color="indianred", alpha=0.4)
    axes[1].plot(dates, dd, color="indianred", linewidth=0.8)
    axes[1].set_title("Drawdown", fontsize=13)
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("Drawdown (ILS)")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    annual.columns = ["Total P&L", "Trades", "Avg P&L"]
    display(annual.round(1))

## Diagnostics

Scatter of trade P&L vs. the underlying's move (spot at exit − spot at entry).
Calendar spreads have a characteristic “tent” shape: profitable when the
underlying stays near the strike, losing when it moves significantly.

In [ ]:
if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    spot_move = results["spot_exit"] - results["spot_entry"]
    pnl = results["pnl"]

    axes[0].scatter(spot_move, pnl, alpha=0.5, s=25, color="steelblue",
                    edgecolors="white", linewidths=0.3)
    axes[0].axhline(0, color="grey", linewidth=0.5, linestyle="--")
    axes[0].axvline(0, color="grey", linewidth=0.5, linestyle="--")
    axes[0].set_title("P&L vs. Spot Move (Entry \u2192 Exit)", fontsize=13)
    axes[0].set_xlabel("Spot Move (ILS)")
    axes[0].set_ylabel("P&L (ILS)")
    axes[0].grid(True, alpha=0.3)

    skip_data = {k: v for k, v in skipped.items() if v > 0}
    if skip_data:
        axes[1].barh(list(skip_data.keys()), list(skip_data.values()),
                     color="slategrey", edgecolor="white")
        axes[1].set_title("Skipped Trades Breakdown", fontsize=13)
        axes[1].set_xlabel("Count")
        axes[1].grid(True, alpha=0.3, axis="x")
    else:
        axes[1].text(0.5, 0.5, "No trades skipped", ha="center", va="center",
                     fontsize=14, color="grey", transform=axes[1].transAxes)
        axes[1].set_title("Skipped Trades Breakdown", fontsize=13)

    fig.tight_layout()
    plt.show()